# Introduction

**Objectives**

Xử lý dữ liệu video quy mô lớn (dạng các file ZIP 3GB) trong môi trường tài nguyên lưu trữ hạn chế (Local Machine, trống ~35GB). Pipeline thực hiện trích xuất Keyframe, chuẩn hóa ID, và định dạng JSONL khép kín theo mô hình "Tải -> Giải nén -> Xử lý -> Xóa".

**Final Dataset**
```text
final_dataset/
├── videos/                  # Chứa file video chuẩn hóa (tùy chọn lưu)
├── keyframes/               # Chứa ảnh frame phân theo video_id
├── videos.jsonl             # Thông tin gốc của video
├── keyframes.jsonl          # Metadata của từng frame
├── temporal_relations.jsonl # Quan hệ thời gian trước/sau
├── validation_report.json   # Báo cáo kiểm định tự động
└── dataset_manifest.json    # Bảng kê khai tổng quan dataset

# Import

In [ ]:
import os
import json
import zipfile
import urllib.request
import shutil
import cv2
from pathlib import Path
from datetime import datetime
import time

# Configs

In [ ]:
# 1. Danh sách ZIP file
ZIP_LINKS = [
    "https://aic-data.ledo.io.vn/Videos_L21_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L22_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L23_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L24_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L25_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L26_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L26_b.zip",
    "https://aic-data.ledo.io.vn/Videos_L26_c.zip",
    "https://aic-data.ledo.io.vn/Videos_L26_d.zip",
    "https://aic-data.ledo.io.vn/Videos_L26_e.zip",
    "https://aic-data.ledo.io.vn/Videos_L27_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L28_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L29_a.zip",
    "https://aic-data.ledo.io.vn/Videos_L30_a.zip"
]

# 2. Cấu hình tham số trích xuất Video
FPS_EXTRACT = 1         # Tốc độ lấy mẫu: 1 frame mỗi giây
DATASET_VERSION = "aic_dataset_v1"
SCHEMA_VERSION = "1.0"

# 3. Cấu hình cấu trúc thư mục làm việc 
WORKSPACE_DIR = Path("./AI_Challenge")
TEMP_PROCESSING_DIR = WORKSPACE_DIR / "temp_processing"
FINAL_DATASET_DIR = WORKSPACE_DIR / "final_dataset"
VIDEOS_DIR = FINAL_DATASET_DIR / "videos"
KEYFRAMES_DIR = FINAL_DATASET_DIR / "keyframes"

# Các file JSONL chuẩn đầu ra
VIDEOS_JSONL = FINAL_DATASET_DIR / "videos.jsonl"
KEYFRAMES_JSONL = FINAL_DATASET_DIR / "keyframes.jsonl"
TEMPORAL_JSONL = FINAL_DATASET_DIR / "temporal_relations.jsonl"

# File lưu tiến độ (Checkpoint)
CHECKPOINT_FILE = WORKSPACE_DIR / "processed_checkpoint.log"

# 4. Tự động khởi tạo cây thư mục
def setup_environment():
    dirs_to_create = [
        TEMP_PROCESSING_DIR, 
        FINAL_DATASET_DIR, 
        VIDEOS_DIR, 
        KEYFRAMES_DIR
    ]
    for d in dirs_to_create:
        d.mkdir(parents=True, exist_ok=True)
    print(f"✅ Môi trường sẵn sàng. Tổng cộng {len(ZIP_LINKS)} file ZIP.")

setup_environment()

✅ Môi trường sẵn sàng. Tổng cộng 14 file ZIP.


# Checkpoint System

In [ ]:
def generate_standard_id(source_name):
    """Chuẩn hóa tên file thành `video_id` mà không cần băm."""
    # Trả về phần tên file không có phần mở rộng (stem), ví dụ: L21_V001
    base = Path(source_name).stem
    return base

def get_processed_indices():
    """Đọc file log để xem những file zip nào đã xử lý xong"""
    if not CHECKPOINT_FILE.exists(): return []
    with open(CHECKPOINT_FILE, 'r') as f:
        return [int(line.strip()) for line in f if line.strip().isdigit()]

def mark_as_processed(index):
    """Ghi lại log sau khi xử lý thành công 1 file zip"""
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(f"{index}\n")


# Core Processing (OpenCV)

In [ ]:
def process_and_extract_video(video_path: Path, video_id: str, fps_extract: int = 1):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise Exception(f"Không thể mở video {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps <= 0 or num_frames <= 0:
        cap.release()
        return 0

    duration_ms = int((num_frames / fps) * 1000)

    # 1. GHI VIDEOS.JSONL
    rel_video_path = f"videos/{video_id}.mp4"
    video_record = {
        "video_id": video_id,
        "video_path": rel_video_path,
        "duration_ms": duration_ms,
        "fps": float(fps),
        "width": width,
        "height": height,
        "num_frames": num_frames
    }
    with open(VIDEOS_JSONL, 'a', encoding='utf-8') as f:
        f.write(json.dumps(video_record) + "\n")

    # 2. TRÍCH XUẤT KEYFRAME & GHI KEYFRAMES.JSONL
    kf_dir = KEYFRAMES_DIR / video_id
    kf_dir.mkdir(parents=True, exist_ok=True)
    
    frame_interval = int(fps / fps_extract) if fps > 0 else 1
    current_frame = 0
    prev_kf_id = None
    window_start_ms = 0
    extracted_count = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        if current_frame % frame_interval == 0:
            timestamp_ms = int((current_frame / fps) * 1000)
            kf_id = f"{video_id}_K{current_frame:06d}"
            rel_kf_path = f"keyframes/{video_id}/{current_frame:06d}.jpg"
            abs_kf_path = FINAL_DATASET_DIR / rel_kf_path

            # Lưu ảnh ra đĩa
            cv2.imwrite(str(abs_kf_path), frame)

            # Ghi keyframes.jsonl
            kf_record = {
                "keyframe_id": kf_id,
                "video_id": video_id,
                "frame_index": current_frame,
                "timestamp_ms": timestamp_ms,
                "keyframe_path": rel_kf_path,
                "shot_id": f"{video_id}_S001"
            }
            with open(KEYFRAMES_JSONL, 'a', encoding='utf-8') as f:
                f.write(json.dumps(kf_record) + "\n")

            # Ghi temporal_relations.jsonl 
            if prev_kf_id is not None:
                temporal_record = {
                    "keyframe_id": prev_kf_id,
                    "previous_keyframe_id": None, 
                    "next_keyframe_id": kf_id,
                    "window_start_ms": window_start_ms,
                    "window_end_ms": timestamp_ms
                }
                with open(TEMPORAL_JSONL, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(temporal_record) + "\n")
            
            window_start_ms = timestamp_ms
            prev_kf_id = kf_id
            extracted_count += 1

        current_frame += 1

    cap.release()
    return extracted_count

# Main Engine

In [ ]:
processed_list = get_processed_indices()

opener = urllib.request.build_opener()
opener.addheaders = [
    ('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'),
    ('Accept', '*/*')
]
urllib.request.install_opener(opener)
# =========================================================

for idx, link in enumerate(ZIP_LINKS):
    if idx in processed_list:
        print(f"⏩ Bỏ qua file {idx} (Đã xử lý).")
        continue
        
    print(f"\n=== ĐANG XỬ LÝ CHUNK {idx + 1}/{len(ZIP_LINKS)} ===")
    zip_path = TEMP_PROCESSING_DIR / f"chunk_{idx}.zip"
    extract_path = TEMP_PROCESSING_DIR / f"extracted_{idx}"
    
    try:
        # 1. TẢI FILE (Lúc này Python đã ẩn danh dưới dạng Chrome)
        print("📥 1. Đang tải ZIP...")
        urllib.request.urlretrieve(link, zip_path)
        
        # 2. GIẢI NÉN
        print("📦 2. Đang giải nén...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
            
        # 3. TÌM VÀ XỬ LÝ MP4
        print("⚙️ 3. Đang trích xuất Keyframe & Metadata...")
        video_files = list(Path(extract_path).rglob("*.mp4"))
        
        if not video_files:
            print("  ⚠️ Không có file MP4 nào bên trong.")
            
        for v_idx, v_path in enumerate(video_files):
            # Lấy tên file gốc
            v_id = generate_standard_id(v_path.name) 
            print(f"  -> Xử lý [{v_idx+1}/{len(video_files)}]: {v_path.name} (ID: {v_id})")
            process_and_extract_video(v_path, v_id, FPS_EXTRACT)
        
        # 4. LƯU TIẾN ĐỘ
        mark_as_processed(idx)
        print(f"✅ Hoàn thành chunk {idx + 1}!")
        
    except Exception as e:
        print(f"❌ LỖI TẠI FILE {idx}: {e}")
        print("Bỏ qua file này, tiếp tục chạy file tiếp theo...")
        continue 
        
    finally:
        # 5. DỌN DẸP RÁC (LUÔN CHẠY DÙ CÓ LỖI HAY KHÔNG)
        print("🧹 4. Dọn dẹp dung lượng tạm...")
        if zip_path.exists(): 
            os.remove(zip_path)
        if extract_path.exists(): 
            shutil.rmtree(extract_path)

# Validator

In [3]:
def validate_dataset():
    print("Bắt đầu kiểm thử toàn vẹn dữ liệu...")
    errors = []
    video_dict = {}
    keyframe_set = set()
    
    if not VIDEOS_JSONL.exists() or not KEYFRAMES_JSONL.exists():
        print("⚠️ Chưa có dữ liệu để kiểm tra!")
        return

    # 1. Kiểm tra videos.jsonl
    with open(VIDEOS_JSONL, 'r', encoding='utf-8') as f:
        for line in f:
            v = json.loads(line)
            if v['video_id'] in video_dict:
                errors.append(f"Trùng lặp video_id: {v['video_id']}")
            video_dict[v['video_id']] = v

    # 2. Kiểm tra keyframes.jsonl
    with open(KEYFRAMES_JSONL, 'r', encoding='utf-8') as f:
        for line in f:
            k = json.loads(line)
            kf_id = k['keyframe_id']
            v_id = k['video_id']
            
            if kf_id in keyframe_set:
                errors.append(f"Trùng lặp keyframe_id: {kf_id}")
            keyframe_set.add(kf_id)
            
            if v_id not in video_dict:
                errors.append(f"Keyframe {kf_id} ánh xạ đến video_id ảo: {v_id}")
                continue
                
            if k['timestamp_ms'] > video_dict[v_id]['duration_ms']:
                errors.append(f"Lỗi logic thời gian: Keyframe {kf_id} vượt quá độ dài video")
                
            if not (FINAL_DATASET_DIR / k['keyframe_path']).exists():
                errors.append(f"Mất file vật lý: Không tìm thấy {k['keyframe_path']}")

    # 3. Ghi Báo cáo
    report = {
        "is_valid": len(errors) == 0,
        "total_errors": len(errors),
        "error_details": errors[:100] 
    }
    with open(FINAL_DATASET_DIR / "validation_report.json", 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2)

    # 4. Ghi Manifest
    if report["is_valid"]:
        manifest = {
          "dataset_version": DATASET_VERSION,
          "schema_version": SCHEMA_VERSION,
          "video_count": len(video_dict),
          "keyframe_count": len(keyframe_set),
          "created_at": datetime.now().isoformat()
        }
        with open(FINAL_DATASET_DIR / "dataset_manifest.json", 'w', encoding='utf-8') as f:
            json.dump(manifest, f, indent=2)
        print("✅ DỮ LIỆU CHUẨN: Validation Passed. Sẵn sàng bàn giao.")
    else:
        print(f"❌ DỮ LIỆU LỖI: Phát hiện {len(errors)} lỗi. Vui lòng check validation_report.json")

validate_dataset()

Bắt đầu kiểm thử toàn vẹn dữ liệu...
✅ DỮ LIỆU CHUẨN: Validation Passed. Sẵn sàng bàn giao.
